<a href="https://colab.research.google.com/github/Wapikkk/text-summarize-nlp/blob/extractive/notebooks/02_extractive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install Sastrawi rouge-score networkx scikit-learn datasets

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import re, json, os, time
import numpy as np
from functools import lru_cache
from google.colab import drive
drive.mount('/content/drive')

def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

Mounted at /content/drive


In [3]:
USE_OFFICIAL = False   # ganti True kalau train/val/test.jsonl dari Peran 1 sudah ada
DATA_DIR = '/content/drive/MyDrive/<folder-drive-bersama>'  # sesuaikan

if USE_OFFICIAL:
    train = read_jsonl(f'{DATA_DIR}/train.jsonl')
    val   = read_jsonl(f'{DATA_DIR}/val.jsonl')
    test  = read_jsonl(f'{DATA_DIR}/test.jsonl')
else:
    from datasets import load_dataset
    ds = load_dataset("esdurmus/wiki_lingua", "indonesian")['train']

    def build_rows(start, end):
        rows = []
        for i in range(start, end):
            art = ds[i]['article']
            for j, (doc, summ) in enumerate(zip(art['document'], art['summary'])):
                rows.append({'id': f'{i}_{j}', 'document': doc, 'summary': summ})
        return rows

    train, val, test = build_rows(0, 600), build_rows(600, 800), build_rows(800, 1000)

print('DATA RESMI' if USE_OFFICIAL else 'DATA SEMENTARA', '|', len(train), len(val), len(test))

README.md:   0%|          | 0.00/23.2k [00:00<?, ?B/s]

indonesian/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 67.7MB            

indonesian/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/16308 [00:00<?, ? examples/s]

DATA SEMENTARA | 1741 586 587


In [4]:
def split_sentences(text):
    text = re.sub(r'\s+', ' ', text).strip()
    sents = re.split(r'(?<=[.!?])\s+(?=[A-Z0-9"“(])', text)
    sents = [s for s in sents if len(s.split()) >= 3]
    return sents or [text]

n_sent  = [len(split_sentences(r['document'])) for r in val]
ref_len = [len(r['summary'].split()) for r in val]
print('kalimat/dokumen (p10, p50, p90):', np.percentile(n_sent,  [10, 50, 90]))
print('kata/ringkasan  (p10, p50, p90):', np.percentile(ref_len, [10, 50, 90]))

kalimat/dokumen (p10, p50, p90): [ 9. 21. 41.]
kata/ringkasan  (p10, p50, p90): [14.  28.  57.5]


In [5]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stemmer = StemmerFactory().create_stemmer()
STOP = set(StopWordRemoverFactory().get_stop_words())

@lru_cache(maxsize=None)
def stem_word(w):
    return stemmer.stem(w)

def prep(sentence, stem=True):
    toks = re.findall(r'[a-z0-9]+', sentence.lower())
    toks = [t for t in toks if t not in STOP]
    return ' '.join(stem_word(t) for t in toks) if stem else ' '.join(toks)

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
import networkx as nx

def pick(scores, sents, k):
    idx = np.argsort(scores)[::-1][:k]
    return ' '.join(sents[i] for i in sorted(idx))

# --- TF-IDF (IDF dihitung dari dokumen train) ---
vec = TfidfVectorizer(norm=None).fit([prep(r['document']) for r in train[:3000]])

def tfidf_summ(doc, k):
    sents = split_sentences(doc)
    if len(sents) <= k: return ' '.join(sents)
    X = vec.transform([prep(s) for s in sents])
    scores = np.asarray(X.sum(axis=1)).ravel() / np.maximum(X.getnnz(axis=1), 1)
    return pick(scores, sents, k)

# --- TextRank ---
def textrank_summ(doc, k):
    sents = split_sentences(doc)
    if len(sents) <= k: return ' '.join(sents)
    try:
        X = TfidfVectorizer().fit_transform([prep(s) for s in sents])
    except ValueError:
        return ' '.join(sents[:k])
    sim = cosine_similarity(X)
    np.fill_diagonal(sim, 0)
    pr = nx.pagerank(nx.from_numpy_array(sim), max_iter=200)
    return pick([pr[i] for i in range(len(sents))], sents, k)

# --- LSA ---
def lsa_summ(doc, k, n_comp=3):
    sents = split_sentences(doc)
    if len(sents) <= k: return ' '.join(sents)
    try:
        X = TfidfVectorizer().fit_transform([prep(s) for s in sents])
    except ValueError:
        return ' '.join(sents[:k])
    nc = min(n_comp, X.shape[0] - 1, X.shape[1] - 1)
    if nc < 1: return ' '.join(sents[:k])
    Z = TruncatedSVD(n_components=nc, random_state=42).fit_transform(X)
    return pick(np.sqrt((Z ** 2).sum(axis=1)), sents, k)

methods = {'tfidf': tfidf_summ, 'textrank': textrank_summ, 'lsa': lsa_summ}

In [7]:
doc = val[0]['document']
print('DOKUMEN  :', doc[:400], '\n')
print('REFERENSI:', val[0]['summary'], '\n')
for name, fn in methods.items():
    print(name.upper(), ':', fn(doc, 2), '\n')

DOKUMEN  : Setelah permintaan maaf Anda diterima, penting bagi kalian berdua untuk melupakan masalah ini dan melanjutkan persahabatan. Masalah yang terus diungkit-ungkit akan menimbulkan lebih banyak percekcokan. Penting bagi Anda dan teman Anda untuk kembali bersenang-senang. Jangan pernah lupa bahwa kalian bersahabat untuk sebuah alasan. Kadang-kadang, setelah bertengkar, Anda butuh waktu untuk mendapatkan 

REFERENSI: Tinggalkan masalah ini di masa lalu. Fokuslah pada hal yang kalian suka. Pelan-pelan sajalah dalam menjalani hubungan. Luangkan waktu untuk fokus pada beberapa hal yang teman Anda suka. 

TFIDF : Masalah yang terus diungkit-ungkit akan menimbulkan lebih banyak percekcokan. Kadang-kadang, setelah bertengkar, Anda butuh waktu untuk mendapatkan kepercayaannya kembali. 

TEXTRANK : Setelah permintaan maaf Anda diterima, penting bagi kalian berdua untuk melupakan masalah ini dan melanjutkan persahabatan. Jangan pernah lupa bahwa kalian bersahabat untuk sebuah alasan. 

LSA 

In [8]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'])

def mean_rouge(preds, refs):
    out = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    for p, r in zip(preds, refs):
        s = scorer.score(r, p)
        for m in out:
            out[m].append(s[m].fmeasure)
    return {m: round(float(np.mean(v)), 4) for m, v in out.items()}

In [9]:
val_s = val[:1000]
refs = [r['summary'] for r in val_s]
print('rata-rata panjang referensi (kata):', round(np.mean([len(x.split()) for x in refs]), 1))

for name, fn in methods.items():
    for k in [1, 2, 3, 4]:
        preds = [fn(r['document'], k) for r in val_s]
        avg_pred = np.mean([len(p.split()) for p in preds])
        print(name, '| k =', k, '| len =', round(avg_pred, 1), '|', mean_rouge(preds, refs))

rata-rata panjang referensi (kata): 32.4
tfidf | k = 1 | len = 17.0 | {'rouge1': 0.1312, 'rouge2': 0.0255, 'rougeL': 0.1059}
tfidf | k = 2 | len = 32.9 | {'rouge1': 0.177, 'rouge2': 0.0376, 'rougeL': 0.132}
tfidf | k = 3 | len = 48.7 | {'rouge1': 0.1936, 'rouge2': 0.0437, 'rougeL': 0.1399}
tfidf | k = 4 | len = 64.0 | {'rouge1': 0.1987, 'rouge2': 0.0475, 'rougeL': 0.1407}
textrank | k = 1 | len = 18.1 | {'rouge1': 0.1937, 'rouge2': 0.0503, 'rougeL': 0.1494}
textrank | k = 2 | len = 35.2 | {'rouge1': 0.2255, 'rouge2': 0.0593, 'rougeL': 0.1662}
textrank | k = 3 | len = 51.4 | {'rouge1': 0.2321, 'rouge2': 0.0622, 'rougeL': 0.1686}
textrank | k = 4 | len = 67.2 | {'rouge1': 0.2271, 'rouge2': 0.0626, 'rougeL': 0.1631}
lsa | k = 1 | len = 14.0 | {'rouge1': 0.1458, 'rouge2': 0.0322, 'rougeL': 0.1171}
lsa | k = 2 | len = 28.5 | {'rouge1': 0.1942, 'rouge2': 0.0455, 'rougeL': 0.1445}
lsa | k = 3 | len = 43.2 | {'rouge1': 0.2099, 'rouge2': 0.0511, 'rougeL': 0.1505}
lsa | k = 4 | len = 57.9 | {'ro

In [10]:
for nc in [2, 3, 5]:
    preds = [lsa_summ(r['document'], 2, n_comp=nc) for r in val_s]
    print('LSA n_comp =', nc, mean_rouge(preds, refs))

LSA n_comp = 2 {'rouge1': 0.1965, 'rouge2': 0.0468, 'rougeL': 0.1456}
LSA n_comp = 3 {'rouge1': 0.1942, 'rouge2': 0.0455, 'rougeL': 0.1445}
LSA n_comp = 5 {'rouge1': 0.1823, 'rouge2': 0.0414, 'rougeL': 0.1385}
